# Stage 05: supervised models

Compares the models that predict species, on the training set only:

| Model | What it sees |
|---|---|
| Chance | nothing: a uniformly random guess |
| Majority class | nothing: always the most common training species |
| Sample rate only | the native sample rate of the recording, nothing about the sound |
| Logistic regression (whole cuts) | mean and standard deviation of each MFCC and delta over the whole cut |
| Logistic regression (windows) | the same statistics over each 3 s window; a cut's windows are averaged |
| LSTM (shuffled frames) | the frames of each 3 s window, in random order |
| LSTM | the frames of each 3 s window, in order |

The two logistic regressions ignore order by construction. The shuffled LSTM is the same model as
the LSTM with the order of the frames destroyed, so the gap between the two measures what order
adds. Each frame's deltas still describe the change over about 40 ms around it, so "order" here
means order beyond that scale.

The test set is used only when `evaluation.evaluate_test` is true in `config.yaml`.

Reads `data/interim/metadata.csv` and `data/features/`. Writes `results/tables/cv_results`,
`results/models/cv_predictions.csv`, `results/models/lstm_learning_curves.csv` and
`results/models/chosen_settings.json`; once the test set is unlocked, also
`results/tables/test_results`, `results/tables/test_gaps` and `results/models/test_predictions.csv`.
Runs in about 12 minutes on a laptop CPU, nearly all of it the LSTM cross-validation.

In [ ]:
# Generated with Claude Code (Anthropic): stage 05, train and compare the supervised models.
import json
import time

import numpy as np
import pandas as pd
from IPython.display import display

from ecoacoustics.config import load_config, resolve_path
from ecoacoustics.evaluation import bootstrap_gap, chance_scores, majority_predictions, scores
from ecoacoustics.features import (
    cut_windows,
    load_clip_stats,
    load_frames,
    seconds_to_frames,
    window_statistics,
)
from ecoacoustics.io import read_metadata
from ecoacoustics.models import (
    fit_predict_logistic,
    fit_predict_rate_tree,
    fit_predict_window_logistic,
    train_lstm,
)
from ecoacoustics.plotting import save_table
from ecoacoustics.splits import filter_species
from ecoacoustics.utils import set_seeds

config = load_config()
SEED = config["seed"]
set_seeds(SEED)
FEATURES, LSTM, EVALUATION = config["features"], config["lstm"], config["evaluation"]
C_GRID, MAX_ITER = config["baseline"]["c_grid"], config["baseline"]["max_iter"]
WINDOW = seconds_to_frames(LSTM["window_seconds"], FEATURES)
HOP = seconds_to_frames(LSTM["window_hop_seconds"], FEATURES)
TABLES_DIR = resolve_path(config, "tables_dir")
MODELS_DIR = resolve_path(config, "models_dir")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## Data

In [ ]:
meta = read_metadata(resolve_path(config, "interim_dir") / "metadata.csv")
features_dir = resolve_path(config, "features_dir")
clip_stats = load_clip_stats(features_dir, FEATURES).set_index("cut_id").loc[meta["cut_id"]]
frames_by_cut = load_frames(features_dir, FEATURES)

# Every array below has one entry per row of metadata.csv, in the same order.
SPECIES = sorted(meta["species"].unique())
N_CLASSES = len(SPECIES)
y = meta["species"].map({name: k for k, name in enumerate(SPECIES)}).to_numpy()
tapes = meta["tape_id"].to_numpy()
rates = meta["sample_rate"].to_numpy()
x_clip = clip_stats.to_numpy()
frames = [frames_by_cut[cut_id] for cut_id in meta["cut_id"]]
x_windows = [window_statistics(cut_windows(f, WINDOW, HOP)) for f in frames]

train = np.flatnonzero(meta["split"] == "train")
test = np.flatnonzero(meta["split"] == "test")
fold_of = meta["cv_fold"].to_numpy()
y_train = y[train]
print(
    f"{len(train)} training cuts on {len(np.unique(tapes[train]))} tapes in "
    f"{len(np.unique(fold_of[train]))} folds; {len(test)} test cuts; {N_CLASSES} species"
)
print(f"Windows of {WINDOW} frames every {HOP} frames: {sum(len(w) for w in x_windows)} in all")

## Cross-validation

The training cuts are already split into 5 folds of whole tapes (stage 02). Each fold is held out
in turn while a fresh model is trained on the other four, so every training cut gets exactly one
prediction from a model that never heard its tape. Scores are computed on these out-of-fold
predictions pooled over all training cuts, so that all 23 species are scored together.

Each model gets one tuned setting: `C`, the inverse strength of the penalty, for the logistic
regressions, and the number of epochs for the LSTMs. The value with the best pooled macro-F1 is
kept. Choosing it on the same predictions makes the cross-validated scores slightly optimistic,
for every model alike; the test set gives the final scores.

In [ ]:
def cross_validate(fit_predict):
    """Out-of-fold probabilities for every training cut.

    ``fit_predict(fit_rows, eval_rows)`` trains a fresh model on ``fit_rows`` and returns
    probabilities for ``eval_rows``, of shape (n_eval, n_classes), or of shape
    (n_settings, n_eval, n_classes) when it tries several settings at once.
    """
    oof = None
    for fold in np.unique(fold_of[train]):
        fit_rows, eval_rows = train[fold_of[train] != fold], train[fold_of[train] == fold]
        probs = fit_predict(fit_rows, eval_rows)
        if oof is None:
            oof = np.zeros((*probs.shape[:-2], len(train), N_CLASSES))
        oof[..., np.searchsorted(train, eval_rows), :] = probs
    return oof


def best_setting(oof_by_setting):
    """Index of the setting with the best pooled macro-F1, and the macro-F1 of every setting."""
    f1 = [scores(y_train, probs.argmax(axis=1))["macro_f1"] for probs in oof_by_setting]
    return int(np.argmax(f1)), f1


cv_rows, cv_predictions, chosen = [], {}, {}


def record(model, setting, oof):
    """Add a model's pooled out-of-fold scores and predictions to the results."""
    cv_predictions[model] = oof.argmax(axis=1)
    cv_rows.append({"Model": model, "Setting": setting, **scores(y_train, oof.argmax(axis=1))})

### Reference models

In [ ]:
rng = np.random.default_rng(SEED)
chance = chance_scores(y_train, np.arange(N_CLASSES), EVALUATION["n_chance_draws"], rng)
cv_rows.append({"Model": "Chance", "Setting": "uniform guess", **chance})

majority = cross_validate(
    lambda fit, ev: np.eye(N_CLASSES)[majority_predictions(y[fit], len(ev))]
)
record("Majority class", "", majority)

rate_tree = cross_validate(
    lambda fit, ev: fit_predict_rate_tree(rates[fit], y[fit], rates[ev], N_CLASSES, SEED)
)
record("Sample rate only", "decision tree", rate_tree)
pd.DataFrame(cv_rows)

### Logistic regression

In [ ]:
lr_clip = cross_validate(
    lambda fit, ev: np.stack(
        [
            fit_predict_logistic(x_clip[fit], y[fit], x_clip[ev], N_CLASSES, c, MAX_ITER)
            for c in C_GRID
        ]
    )
)
lr_windows = cross_validate(
    lambda fit, ev: np.stack(
        [
            fit_predict_window_logistic(
                [x_windows[i] for i in fit],
                y[fit],
                [x_windows[i] for i in ev],
                N_CLASSES,
                c,
                MAX_ITER,
            )
            for c in C_GRID
        ]
    )
)
for name, oof in [
    ("Logistic regression (whole cuts)", lr_clip),
    ("Logistic regression (windows)", lr_windows),
]:
    best, f1 = best_setting(oof)
    chosen[name] = {"C": C_GRID[best]}
    by_c = {c: round(v, 3) for c, v in zip(C_GRID, f1, strict=True)}
    print(name, "- pooled macro-F1 by C:", by_c)
    record(name, f"C = {C_GRID[best]}", oof[best])

### LSTM and its shuffled-order control

Both are trained with the same seed, so they start from the same weights and see the same windows;
the only difference is the order of the frames. This cell takes about 10 minutes.

In [ ]:
def lstm_cross_validate(shuffle_order):
    return cross_validate(
        lambda fit, ev: train_lstm(
            [frames[i] for i in fit],
            y[fit],
            [frames[i] for i in ev],
            N_CLASSES,
            LSTM,
            WINDOW,
            HOP,
            LSTM["max_epochs"],
            shuffle_order,
            SEED,
        )
    )


lstm_runs = {}
for name, shuffle_order in [("LSTM (shuffled frames)", True), ("LSTM", False)]:
    start = time.perf_counter()
    lstm_runs[name] = lstm_cross_validate(shuffle_order)
    best, f1 = best_setting(lstm_runs[name])
    chosen[name] = {"epochs": best + 1}
    record(name, f"{best + 1} epochs", lstm_runs[name][best])
    print(
        f"{name}: best pooled macro-F1 {f1[best]:.3f} after {best + 1} epochs "
        f"({time.perf_counter() - start:.0f} s)"
    )

### Results

In [ ]:
cv_table = pd.DataFrame(cv_rows).rename(columns={"macro_f1": "Macro-F1", "accuracy": "Accuracy"})
save_table(cv_table, "cv_results", TABLES_DIR, "%.3f")

curves = pd.DataFrame(
    [
        {"model": name, "epoch": epoch + 1, **scores(y_train, oof.argmax(axis=1))}
        for name, run in lstm_runs.items()
        for epoch, oof in enumerate(run)
    ]
)
curves.to_csv(MODELS_DIR / "lstm_learning_curves.csv", index=False)

predicted = {model: [SPECIES[k] for k in pred] for model, pred in cv_predictions.items()}
pd.DataFrame(
    {
        "cut_id": meta["cut_id"].to_numpy()[train],
        "tape_id": tapes[train],
        "species": meta["species"].to_numpy()[train],
        **predicted,
    }
).to_csv(MODELS_DIR / "cv_predictions.csv", index=False)
(MODELS_DIR / "chosen_settings.json").write_text(json.dumps(chosen, indent=2), encoding="utf-8")
cv_table

## Test set

Every model is trained once on all training cuts with the settings chosen above, then scored on
the test set. Three gaps in macro-F1 get a bootstrap interval that resamples whole test tapes:

- LSTM minus logistic regression on the same windows: order and model together, same input.
- LSTM minus logistic regression on whole cuts: hypothesis H1 as stated in the proposal.
- LSTM minus LSTM on shuffled frames: what order adds on its own.

The comparison is then repeated with only the cuts recorded at 16 kHz or above, keeping the
species that still pass the stage 02 thresholds, to check that the results do not rest on the
empty high frequencies of the low-rate recordings.

In [ ]:
COMPARISONS = [
    ("LSTM", "Logistic regression (windows)", "Order and model, same windows"),
    ("LSTM", "Logistic regression (whole cuts)", "H1 as stated"),
    ("LSTM", "LSTM (shuffled frames)", "Order alone"),
]


def final_predictions(fit_rows, eval_rows):
    """Train every model once on ``fit_rows`` with the chosen settings and predict ``eval_rows``."""
    clip_c = chosen["Logistic regression (whole cuts)"]["C"]
    window_c = chosen["Logistic regression (windows)"]["C"]
    predictions = {
        "Majority class": majority_predictions(y[fit_rows], len(eval_rows)),
        "Sample rate only": fit_predict_rate_tree(
            rates[fit_rows], y[fit_rows], rates[eval_rows], N_CLASSES, SEED
        ).argmax(axis=1),
        "Logistic regression (whole cuts)": fit_predict_logistic(
            x_clip[fit_rows], y[fit_rows], x_clip[eval_rows], N_CLASSES, clip_c, MAX_ITER
        ).argmax(axis=1),
        "Logistic regression (windows)": fit_predict_window_logistic(
            [x_windows[i] for i in fit_rows],
            y[fit_rows],
            [x_windows[i] for i in eval_rows],
            N_CLASSES,
            window_c,
            MAX_ITER,
        ).argmax(axis=1),
    }
    for name, shuffle_order in [("LSTM (shuffled frames)", True), ("LSTM", False)]:
        history = train_lstm(
            [frames[i] for i in fit_rows],
            y[fit_rows],
            [frames[i] for i in eval_rows],
            N_CLASSES,
            LSTM,
            WINDOW,
            HOP,
            chosen[name]["epochs"],
            shuffle_order,
            SEED,
        )
        predictions[name] = history[-1].argmax(axis=1)
    return predictions

In [ ]:
if not EVALUATION["evaluate_test"]:
    print(
        "The test set is locked. Set evaluation.evaluate_test to true in config.yaml once the "
        "settings above are final; the test set is then used once."
    )
else:
    prep = config["prepare"]
    min_rate = EVALUATION["robustness_min_sample_rate"]
    kept, _ = filter_species(
        meta[meta["sample_rate"] >= min_rate],
        prep["grouping_key"],
        prep["min_cuts_per_species"],
        prep["min_groups_per_species"],
    )
    robust = meta.index.isin(kept.index)
    data_sets = {
        "All cuts": (train, test),
        f"Cuts at {min_rate} Hz or more": (train[robust[train]], test[robust[test]]),
    }
    rng = np.random.default_rng(SEED)
    test_rows, gap_rows, saved = [], [], []
    for data_name, (fit_rows, eval_rows) in data_sets.items():
        y_eval = y[eval_rows]
        predictions = final_predictions(fit_rows, eval_rows)
        chance = chance_scores(
            y_eval, np.unique(y[fit_rows]), EVALUATION["n_chance_draws"], rng
        )
        test_rows.append({"Data": data_name, "Model": "Chance", **chance})
        for model, pred in predictions.items():
            test_rows.append({"Data": data_name, "Model": model, **scores(y_eval, pred)})
        for model_a, model_b, question in COMPARISONS:
            gap = bootstrap_gap(
                y_eval,
                predictions[model_a],
                predictions[model_b],
                tapes[eval_rows],
                EVALUATION["n_bootstrap"],
                EVALUATION["ci_level"],
                rng,
            )
            gap_rows.append(
                {"Data": data_name, "Comparison": f"{model_a} minus {model_b}",
                 "Question": question, **gap}
            )
        saved.append(
            pd.DataFrame(
                {
                    "data": data_name,
                    "cut_id": meta["cut_id"].to_numpy()[eval_rows],
                    "tape_id": tapes[eval_rows],
                    "species": meta["species"].to_numpy()[eval_rows],
                    **{m: [SPECIES[k] for k in p] for m, p in predictions.items()},
                }
            )
        )
    test_table = pd.DataFrame(test_rows).rename(
        columns={"macro_f1": "Macro-F1", "accuracy": "Accuracy"}
    )
    gap_table = pd.DataFrame(gap_rows).rename(
        columns={
            "gap": "Macro-F1 gap",
            "ci_low": "CI low",
            "ci_high": "CI high",
            "share_a_better": "Share of resamples with first model better",
        }
    )
    save_table(test_table, "test_results", TABLES_DIR, "%.3f")
    save_table(gap_table, "test_gaps", TABLES_DIR, "%.3f")
    pd.concat(saved).to_csv(MODELS_DIR / "test_predictions.csv", index=False)
    display(test_table)
    display(gap_table)